In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['CRYPTOGRAPHY_OPENSSL_NO_LEGACY'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

import datasets
from datasets import load_dataset, Dataset, DatasetDict, Dataset
from tqdm.auto import tqdm
import json

datasets.disable_caching()

/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/miniconda3/envs/megatron-next/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from megatron.energon import get_train_dataset, WorkerConfig, get_loader, DefaultTaskEncoder, Cooker, basic_sample_keys
from megatron.energon import DefaultTaskEncoder, TextSample, Sample
import gzip
import pickle
import webdataset as wds
import json
import dataclasses
import torch

from tqdm.auto import tqdm
from typing import List

from qwen3_base_energon_helpers import MyTaskEncoder, print_error_handler

In [3]:
tokenizer_path="/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/code/Megatron-Next/model_dir/pulse_v19_80b_a3b_next_gemini_bf16/tokenizer_v19_gemini"

In [4]:
worker_config = WorkerConfig(
    rank=0,
    world_size=1,
    seed_offset=123456789,
    num_workers=1,
)

train_ds = get_train_dataset(
    '/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/code/Megatron-Next/model_dir/pulse_v19_80b_a3b_next_gemini_bf16/dataset.yaml',
    split_part="train",
    task_encoder=MyTaskEncoder(
        tokenizer_path=tokenizer_path,
        seq_length=66536,
        sensitive_words_path=None,
    ),
    batch_size=1,
    packing_buffer_size=None,
    shuffle_buffer_size=None,
    max_samples_per_sequence=None,
    worker_config=worker_config,
)

train_dataloader = get_loader(train_ds, worker_config=worker_config,)

rank=0, worker=0: sample_range=[0, 98994] in 1 slices, sum(count)=98994: indexes=[0, 98994] slices=[train-chunk-0.tar[0(start), 98994(end)]]
rank=0, worker=0: sample_range=[0, 33580] in 1 slices, sum(count)=33580: indexes=[0, 33580] slices=[train-chunk-0.tar[0(start), 33580(end)]]
rank=0, worker=0: sample_range=[0, 70519] in 1 slices, sum(count)=70519: indexes=[0, 70519] slices=[train-chunk-0.tar[0(start), 70519(end)]]
rank=0, worker=0: sample_range=[0, 99256] in 1 slices, sum(count)=99256: indexes=[0, 99256] slices=[train-chunk-0.tar[0(start), 99256(end)]]
rank=0, worker=0: sample_range=[0, 14160] in 1 slices, sum(count)=14160: indexes=[0, 14160] slices=[train-chunk-0.tar[0(start), 14160(end)]]
rank=0, worker=0: sample_range=[0, 5231] in 1 slices, sum(count)=5231: indexes=[0, 5231] slices=[train-chunk-0.tar[0(start), 5231(end)]]
rank=0, worker=0: sample_range=[0, 1453] in 1 slices, sum(count)=1453: indexes=[0, 1453] slices=[train-chunk-0.tar[0(start), 1453(end)]]
rank=0, worker=0: sam

/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/miniconda3/envs/megatron-next/lib/python3.12/site-packages/megatron/energon/loader.py:108: FutureWarning: Passing a worker_config to get_loader() is deprecated and will have no effect.
  warn_deprecated(


In [5]:
dd = iter(train_dataloader)

In [6]:
import json

count_map = {}

with open("/inspire/hdd/global_user/zhangxiaofan-25025/genggui001/code/Megatron-Next/model_dir/pulse_v19_80b_a3b_next_gemini_bf16/qua/pulse_v19_32b_gemini_train.jsonl", "w", encoding="utf8") as f:
    for _ in tqdm(range(2048)):
        item = next(dd)
        
        kk = item['__key__'][0].split("/")[-1].split("-train-")[0]
        if kk not in count_map:
            count_map[kk] = 0
        
        count_map[kk] += 1
              
        item = {
            "input_ids": item['input_ids'].tolist()[0],
            "labels": item['labels'].tolist()[0],
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        

            
        

100%|██████████| 2048/2048 [01:52<00:00, 18.17it/s] 


Token indices sequence length is longer than the specified maximum sequence length for this model (67720 > 66536). Running this sequence through the model will result in indexing errors


In [7]:
count_list = sorted(count_map.items(), key=lambda x:-x[1])
[(k,v/2048)for k,v in count_list]

[('gg_en_fineweb-edu_data', 0.07763671875),
 ('gg_zh_gg_zh_edu_data', 0.07177734375),
 ('zh_med_our_med_kb', 0.05517578125),
 ('gg_zh_med_ns_data', 0.0419921875),
 ('evol-codealpaca-v1', 0.03515625),
 ('exam-math-cn', 0.025390625),
 ('function_call', 0.0234375),
 ('预问诊', 0.0224609375),
 ('aya_dataset', 0.02197265625),
 ('exam-en', 0.0185546875),
 ('multi_my_function_tools', 0.0185546875),
 ('gg_en_med_ns_data', 0.01806640625),
 ('WildChat', 0.017578125),
 ('general-cn-my', 0.01708984375),
 ('tulu-3-IF-augmented', 0.015625),
 ('OpenMathInstruct-2', 0.01513671875),
 ('longalign-en-think', 0.01513671875),
 ('DeepMath-103K-same-language-think', 0.0126953125),
 ('zh_ChinaNews-cn_data', 0.01171875),
 ('导诊考试', 0.01171875),
 ('tulu-3-sft-personas-code', 0.01171875),
 ('general_multilingual_base-en-think', 0.0107421875),
 ('exam-science-cn', 0.01025390625),
 ('general_en_base-same-language-think', 0.01025390625),
 ('NuminaMath-TIR', 0.01025390625),
 ('导诊', 0.009765625),
 ('evol-instruct-long-en

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

In [9]:
tokenizer("<|im_start|>assistant\n<think>\n").input_ids 

[151644, 77091, 198, 13708, 766, 397]

In [12]:
tokenizer("<|im_start|>assistant\n<ggthink>\n").input_ids 

[151644, 77091, 198, 151667, 198]

In [26]:
item = next(dd)
# print(item['__key__'][0].split("/")[-1].split("-train-")[0])
print(tokenizer.decode([item for item in item['input_ids'].tolist()[0] if item != -100]))

<|im_start|>user
Find $a$ and $b$ if

\begin{align*}
a^2+2b^2&=8,\\
a^3+3b^3 &=26.
\end{align*}<|im_end|>
<|im_start|>assistant
<ggthink>

</ggthink>

Let the given system of equations be
\begin{align*} \label{eq:1} a^2+2b^2&=8 \\ a^3+3b^3 &=26\end{align*}
We are looking for real values of $a$ and $b$.

The first equation $a^2+2b^2=8$ describes an ellipse centered at the origin in the $(a,b)$-plane. The semi-axes are $\sqrt{8}=2\sqrt{2}$ along the $a$-axis and $\sqrt{8/2}=2$ along the $b$-axis. The points $(a,b)$ satisfying this equation are bounded, specifically $-2\sqrt{2} \le a \le 2\sqrt{2}$ and $-2 \le b \le 2$.

We want to find the range of the function $F(a,b) = a^3+3b^3$ subject to the constraint $g(a,b) = a^2+2b^2-8=0$. We can use the method of Lagrange multipliers. Let $\nabla F = \lambda \nabla g$.
The partial derivatives are:
$\frac{\partial F}{\partial a} = 3a^2$, $\frac{\partial F}{\partial b} = 9b^2$
$\frac{\partial g}{\partial a} = 2a$, $\frac{\partial g}{\partial b} = 

In [27]:
print(tokenizer.decode([item for item in item['labels'].tolist()[0] if item != -100]))

Let the given system of equations be
\begin{align*} \label{eq:1} a^2+2b^2&=8 \\ a^3+3b^3 &=26\end{align*}
We are looking for real values of $a$ and $b$.

The first equation $a^2+2b^2=8$ describes an ellipse centered at the origin in the $(a,b)$-plane. The semi-axes are $\sqrt{8}=2\sqrt{2}$ along the $a$-axis and $\sqrt{8/2}=2$ along the $b$-axis. The points $(a,b)$ satisfying this equation are bounded, specifically $-2\sqrt{2} \le a \le 2\sqrt{2}$ and $-2 \le b \le 2$.

We want to find the range of the function $F(a,b) = a^3+3b^3$ subject to the constraint $g(a,b) = a^2+2b^2-8=0$. We can use the method of Lagrange multipliers. Let $\nabla F = \lambda \nabla g$.
The partial derivatives are:
$\frac{\partial F}{\partial a} = 3a^2$, $\frac{\partial F}{\partial b} = 9b^2$
$\frac{\partial g}{\partial a} = 2a$, $\frac{\partial g}{\partial b} = 4b$

The Lagrange multiplier equations are:
1) $3a^2 = \lambda (2a)$
2) $9b^2 = \lambda (4b)$
3) $a^2+2b^2=8$

From equation (1), we have two possibili